# Análisis de Estacionalidad de Pernoctaciones
**Analista: Rubén Serra Sanz | 11 Mayo 2026**

In [2]:
import pandas as pd

df_analisis_temp = pd.read_csv(r"C:\Simulador_Análisis_datos\Sprint 4\cleanned_pernoctaciones_temporal_11_05.csv")
df_analisis_temp.groupby('Comunidad Autonoma')['Total'].describe()


,count,mean,std,min,25%,50%,75%,max
Comunidad Autonoma,,,,,,,,
Andalucía,51.0,4.460732e+06,1.726290e+06,1463661.0,2834582.0,4820554.0,5768920.5,7586225.0
Baleares,51.0,4.776598e+06,4.316498e+06,127429.0,424215.0,3417164.0,9025930.5,11414856.0
C. Valenciana,51.0,2.440850e+06,8.593816e+05,713563.0,1664684.5,2526943.0,2932227.0,4078805.0
Cataluña,51.0,4.739988e+06,2.243013e+06,1461340.0,2631259.0,4661134.0,6488953.0,8900068.0
Madrid,51.0,2.121953e+06,2.799654e+05,1193236.0,1961731.0,2165912.0,2308151.0,2598869.0
Total,51.0,2.855204e+07,1.077883e+07,10598385.0,18460214.0,28046754.0,37117627.0,48181913.0


In [3]:
import plotly.express as px
import pandas as pd

# 1. Diccionario de colores coherente
color_map = {
    'Andalucía': '#E9C46A',      # Amarillo Azafrán
    'Baleares': '#7A919E',       # Azul Pizarra
    'C. Valenciana': '#CD8D7A',  # Terracota Pastel
    'Cataluña': '#8E9D7D',       # Verde Salvia
    'Madrid': '#D4A373',         # Ocre Suave
    'Resto de España': '#D3D3D3' # Gris Claro
}

# 2. Preparación de datos
df_solo_ccaa = df_analisis_temp[df_analisis_temp['Comunidad Autonoma'] != 'Total'].copy()
df_solo_ccaa['Fecha'] = pd.to_datetime(df_solo_ccaa['Fecha'])

# 3. Creación del gráfico
fig_ccaa = px.line(
    df_solo_ccaa, 
    x='Fecha', 
    y='Total', 
    color='Comunidad Autonoma',
    color_discrete_map=color_map,
    template='plotly_white'
)

# AUMENTO DE GROSOR DE LÍNEAS (Width=4 es un grosor elegante pero potente)
fig_ccaa.update_traces(line=dict(width=4))

# Formato de meses y lógica de filtrado de etiquetas
meses_es = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
fechas_unicas = sorted(df_solo_ccaa['Fecha'].unique())

# Filtramos para mostrar solo una etiqueta cada 3 meses (Trimestral)
fechas_filtradas = fechas_unicas[::3] 

# 4. Ajuste de diseño final
fig_ccaa.update_layout(
    width=1200,
    height=700, 
    showlegend=False,
    paper_bgcolor='rgba(0,0,0,0)', # Fondo transparente
    plot_bgcolor='rgba(0,0,0,0)',  # Fondo transparente
    xaxis=dict(
        title_text='',
        showgrid=False,
        zeroline=False,
        tickangle=-45,
        tickfont=dict(size=18, color='#444'),
        tickvals=fechas_filtradas, # Solo mostramos las seleccionadas
        ticktext=[f"{meses_es[pd.to_datetime(d).month-1]} {pd.to_datetime(d).year}" for d in fechas_filtradas]
    ),
    yaxis=dict(
        title_text='',
        showgrid=False,
        zeroline=False,
        tickfont=dict(size=18, color='#444')
    ),
    margin=dict(b=150, t=50, l=100, r=40), 
    hovermode='x unified'
)

fig_ccaa.show()

### Conclusiones: Evolución de Pernoctaciones por CCAA


- Baleares como motor estival y riesgo operativo: Presenta los picos más altos de toda la serie histórica en los meses de verano (rozando los 11M-12M de pernoctaciones), 
pero sufre la caída más dramática e inmediata del gráfico al llegar el invierno, llegando casi a cero. 
Es el comportamiento más volátil del portafolio.

- Cataluña y Andalucía (Resiliencia Masiva): Siguen la tendencia estacional del verano, pero sus valles de invierno son notablemente más altos y seguros (manteniéndose por encima de los 2M). Muestran una base de demanda mucho más sólida durante todo el año.

- Madrid como estabilizador de ingresos: Es la única línea plana y constante del gráfico. No sufre por el factor "sol y playa", lo que convierte a esta región en el pulmón financiero que sostiene los costes fijos de la empresa durante los meses fríos

In [27]:
import plotly.express as px
import pandas as pd

# 1. Mapa de colores Neutro (con Andalucía en Amarillo Azafrán)
color_map = {
    'Andalucía': '#E9C46A',      # Amarillo Azafrán
    'Baleares': '#7A919E',       # Azul Pizarra
    'C. Valenciana': '#CD8D7A',  # Terracota Pastel
    'Cataluña': '#8E9D7D',       # Verde Salvia
    'Madrid': '#D4A373',         # Ocre Suave
    'Resto de España': '#D3D3D3' # Gris Claro
}

# 2. Preparación de datos por MES
df_mensual = df_analisis_temp.copy()
df_mensual['Fecha'] = pd.to_datetime(df_mensual['Fecha'])

# Extraemos el número del mes para ordenar y el nombre para mostrar
df_mensual['Mes_Num'] = df_mensual['Fecha'].dt.month
df_mensual['Mes_Nombre'] = df_mensual['Fecha'].dt.month_name(locale='es_ES') 

# Separamos CCAA y Total Nacional
df_solo_ccaa = df_mensual[df_mensual['Comunidad Autonoma'] != 'Total'].copy()
df_total_nacional = df_mensual[df_mensual['Comunidad Autonoma'] == 'Total'].copy()

# Agrupamos por mes (promediando o sumando según tus datos, aquí sumamos)
resumen_ccaa = df_solo_ccaa.groupby(['Mes_Num', 'Mes_Nombre', 'Comunidad Autonoma'])['Total'].sum().reset_index()
resumen_total = df_total_nacional.groupby(['Mes_Num', 'Mes_Nombre'])['Total'].sum().reset_index()

# Calculamos el "Resto de España" mensual
total_vuestras = resumen_ccaa.groupby(['Mes_Num', 'Mes_Nombre'])['Total'].sum().reset_index()
df_resto = pd.merge(total_vuestras, resumen_total, on=['Mes_Num', 'Mes_Nombre'], suffixes=('_Vuestras', '_Nacional'))
df_resto['Total'] = df_resto['Total_Nacional'] - df_resto['Total_Vuestras']
df_resto['Comunidad Autonoma'] = 'Resto de España'

# Unimos todo para el cálculo final de porcentaje
df_completo = pd.concat([resumen_ccaa, df_resto[['Mes_Num', 'Mes_Nombre', 'Comunidad Autonoma', 'Total']]])
df_final = pd.merge(df_completo, resumen_total, on=['Mes_Num', 'Mes_Nombre'], suffixes=('', '_Nacional'))
df_final['Porcentaje_Real'] = (df_final['Total'] / df_final['Total_Nacional']) * 100

# Ordenamos por número de mes para que el gráfico vaya de Enero a Diciembre
df_final = df_final.sort_values('Mes_Num')

# 3. Creación del gráfico de barras apiladas al 100%
fig_market_share = px.bar(
    df_final, 
    x='Mes_Nombre', 
    y='Porcentaje_Real', 
    color='Comunidad Autonoma',
    color_discrete_map=color_map,
    category_orders={"Comunidad Autonoma": ["Andalucía", "Baleares", "C. Valenciana", "Cataluña", "Madrid", "Resto de España"]},
    text=df_final['Porcentaje_Real'].apply(lambda x: f'{x:.1f}%' if x > 4 else ''), # Solo muestra texto si cabe (>4%)
    template='plotly_white'
)

# 4. Ajustes finales (Minimalistas y Corporativos)
fig_market_share.update_layout(
    width=1700,
    height=750,
    bargap=0.2,
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    showlegend=False,
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.1,
        xanchor="center",
        x=1,
        title_text=''
    ),
    xaxis=dict(
        title_text='', 
        tickfont=dict(size=18),
        showline=True, 
        linewidth=2, 
        linecolor='#444'
    ),
    yaxis=dict(
        title_text='Peso sobre Total Nacional (%)',
        title_font=dict(size=18),
        range=[0, 100],
        showgrid=False,
        showline=True, 
        linewidth=2, 
        linecolor='#444',
        ticksuffix='%',
        tickfont=dict(size=18)
    ),
    margin=dict(b=100, t=50, l=80, r=40),
    barmode='stack'
)

fig_market_share.update_traces(textposition='inside', textfont_size=18)

fig_market_share.show()

### Conclusiones distribución de la Demanda de pernoctaciones nacional

- Baleares: Es la comunidad que más altera la distribución a lo largo del año. En invierno representa una cuota residual (apenas un 1.2% o 1.9% en enero/febrero), pero en los meses de verano (junio a septiembre) absorbe de golpe hasta una cuarta parte (25%) de todo el volumen nacional.

- Andalucía como líder en estabilidad: Mantiene una cuota masiva y constante durante todo el año, moviéndose siempre en un canal muy sólido entre el 13.9% (enero) y el 17.4% (abril). Es el destino con menor riesgo del portafolio.

- Cataluña y Comunidad Valenciana como pilares estivales secundarios: Cataluña dobla su peso al llegar el verano (pasa de un 14.2% en enero a un 18.7% en agosto). La Comunidad Valenciana, aunque con porcentajes más discretos (en torno al 8%-9.9%), exhibe un comportamiento muy estable mes a mes.

- Madrid: Al ser turismo urbano/corporativo, su mayor cuota del pastel se da en los meses de otoño e invierno (alcanzando el 11.4% o 11.9% en noviembre/diciembre), mientras que en pleno verano cae a su mínimo histórico (4.2% en agosto).

In [29]:
import plotly.express as px
import pandas as pd

# 1. Tu mapa de colores oficial
color_map = {
    'Andalucía': '#7e3ff2',
    'Baleares': '#5470ff',
    'C. Valenciana': '#ff5a3c',
    'Cataluña': '#00cc96',
    'Madrid': '#ffa35c'
}

# Diccionario de traducción
meses_es = {
    1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril',
    5: 'Mayo', 6: 'Junio', 7: 'Julio', 8: 'Agosto',
    9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
}

# 2. Preparación de datos
df_promedio = df_analisis_temp[df_analisis_temp['Comunidad Autonoma'] != 'Total'].copy()
df_promedio['Fecha'] = pd.to_datetime(df_promedio['Fecha'])
df_promedio['Mes_Num'] = df_promedio['Fecha'].dt.month
df_promedio['Mes_Nombre'] = df_promedio['Mes_Num'].map(meses_es)

resumen_media = df_promedio.groupby(['Mes_Num', 'Mes_Nombre', 'Comunidad Autonoma'])['Total'].mean().reset_index()
resumen_media = resumen_media.sort_values('Mes_Num')

# 3. Creación del gráfico
fig_estacionalidad = px.bar(
    resumen_media,
    x='Mes_Nombre',
    y='Total',
    color='Comunidad Autonoma',
    barmode='group',
    color_discrete_map=color_map,
    category_orders={
        "Mes_Nombre": list(meses_es.values()),
        "Comunidad Autonoma": ["Andalucía", "Baleares", "C. Valenciana", "Cataluña", "Madrid"]
    },
    template='plotly_white'
)

# 4. Ajustes estéticos (GRANDE, ANCHO Y CORREGIDO)
fig_estacionalidad.update_layout(
    title_text='', 
    width=1600,
    height=700,
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    showlegend=False,
    xaxis=dict(
        title_text='',
        showgrid=False,
        showline=True,
        linewidth=2,
        linecolor='#444',
        tickfont=dict(size=14)
    ),
    yaxis=dict(
        title_text='Media de Pernoctaciones',
        # Corregido: title_font en lugar de titlefont
        title_font=dict(size=16), 
        showgrid=True,
        gridcolor='#eeeeee',
        showline=True,
        linewidth=2,
        linecolor='#444',
        tickfont=dict(size=12)
    ),
    bargap=0.15,
    bargroupgap=0.1,
    margin=dict(b=60, t=20, l=100, r=40)
)

fig_estacionalidad.show()

### Análisis concentración de la Demanda (Índice de Gini)

In [30]:
import numpy as np

def gini(array):
    """Calcula el coeficiente de Gini de una lista de valores."""
    array = np.array(array, dtype=np.float64)
    if np.amin(array) < 0:
        array -= np.amin(array) # Valores deben ser no negativos
    array += 0.0000001 # Evitar división por cero
    array = np.sort(array)
    index = np.arange(1, array.shape[0] + 1)
    n = array.shape[0]
    return ((np.sum((2 * index - n - 1) * array)) / (n * np.sum(array)))

# 1. Preparar los datos
df_gini = df_analisis_temp[df_analisis_temp['Comunidad Autonoma'] != 'Total'].copy()
df_gini['Año'] = pd.to_datetime(df_gini['Fecha']).dt.year

# 2. Calcular Gini por CCAA y Año
resultados_gini = []

for (ccaa, año), grupo in df_gini.groupby(['Comunidad Autonoma', 'Año']):
    # Necesitamos los 12 meses para que el índice sea válido
    if len(grupo) == 12: 
        valor_gini = gini(grupo['Total'].values)
        resultados_gini.append({'CCAA': ccaa, 'Año': año, 'Gini': valor_gini})
    elif año == 2026: # Para el año actual (incompleto) podemos avisar
        valor_gini = gini(grupo['Total'].values)
        resultados_gini.append({'CCAA': ccaa, 'Año': año, 'Gini': valor_gini})

df_resultados = pd.DataFrame(resultados_gini)

# 3. Mostrar como tabla pivote para comparar años
tabla_gini = df_resultados.pivot(index='CCAA', columns='Año', values='Gini')
print(tabla_gini)

Año                2022      2023      2024      2025      2026
CCAA                                                           
Andalucía      0.247237  0.211112  0.194624  0.202440  0.110519
Baleares       0.495210  0.475922  0.461471  0.461439  0.364289
C. Valenciana  0.235888  0.187534  0.176828  0.177745  0.114430
Cataluña       0.287505  0.252090  0.250636  0.246042  0.070796
Madrid         0.080800  0.050001  0.050011  0.051849  0.029772


In [47]:
import plotly.express as px

# 1. Tu mapa de colores personalizado
color_map = {
    'Andalucía': '#E9C46A',    # Amarillo Azafrán
    'Baleares': '#7A919E',     # Azul Pizarra
    'C. Valenciana': '#CD8D7A', # Terracota Pastel
    'Cataluña': '#8E9D7D',     # Verde Salvia
    'Madrid': '#D4A373'        # Ocre / Arena
}

# 2. Filtrar datos (2022-2025)
df_historico = df_resultados[df_resultados['Año'].isin([2022, 2023, 2024, 2025])].copy()

# 3. Creación del gráfico
fig_gini_final = px.bar(
    df_historico, 
    x='Año', 
    y='Gini', 
    color='CCAA',
    barmode='group',
    color_discrete_map=color_map,
    text=df_historico['Gini'].apply(lambda x: f'{x:.2f}'),
    template='plotly_dark' # Fondo oscuro como en tu captura
)

# 4. Ajustes estéticos y de fuente
fig_gini_final.update_layout(
    width=1300,
    height=750,
    showlegend=True,
    paper_bgcolor='rgba(0,0,0,0)', # Transparente para que pegue en tu slide
    plot_bgcolor='rgba(0,0,0,0)',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.05,
        xanchor="center",
        x=0.5,
        title_text='',
        font=dict(size=16)
    ),
    xaxis=dict(
        title_text='',
        tickmode='linear',
        tickfont=dict(size=18, color='white'),
        showgrid=False
    ),
    yaxis=dict(
        title_text='Índice de Gini (Concentración)',
        title_font=dict(size=18, color='black'),
        tickfont=dict(size=16, color='black'),
        gridcolor='rgba(255,255,255,0.1)', # Rejilla sutil
        range=[0, 0.6] # Ajustado al máximo de Baleares
    ),
    margin=dict(b=80, t=120, l=100, r=40)
)

fig_gini_final.update_traces(
    textposition='outside', 
    textfont_size=14,
    marker_line_width=0
)

fig_gini_final.show()

### Conclusiones: Evolución del Índice de Gini (Concentración Estacional)

- Baleares como el caso crítico y estructural: Registra, con una diferencia abismal, el Índice de Gini más alto de todo el estudio de forma consecutiva (0.50 en 2022 y bajando levemente a 0.46 en 2025). Esto demuestra de forma matemática y rigurosa lo que observamos en los volúmenes: su negocio está hiperconcentrado en los meses de verano, mostrando una resistencia estructural a la desestacionalización natural.

- Madrid como el modelo de estabilidad perfecta: En el extremo opuesto, Madrid presenta un índice prácticamente nulo (0.08 en 2022 y cayendo a 0.05 en 2025). Esto confirma que el flujo de visitantes en la capital es completamente homogéneo y lineal durante los 12 meses del año, libre de riesgos estacionales.

- Andalucía ha mejorado notablemente la distribución de su demanda, reduciendo su índice de 0.25 a 0.20.

- Comunidad Valenciana se mantiene como una de las zonas de costa más estables (0.18).

- Cataluña muestra una concentración intermedia-alta y constante en los 4 años (0.25-0.29) debido a su fuerte peso de turismo estival de sol y playa.